In [29]:
import pandas as pd

actual_wind = pd.read_csv("actual-wind generation.csv")
day_ahead   = pd.read_csv("day-ahead price.csv")
intraday    = pd.read_csv("intraprice.csv")
water       = pd.read_csv("water.csv")
wind_fc     = pd.read_csv("wind_forecast.csv")

In [33]:
# Change the time to local time but without timesaving

def parse_local_time(s):
    s = s.astype(str)
    is_second = s.str.contains(r"\s2B:")             # second pass through 02:00-03:00
    s = s.str.replace(r"2[AB]:", "02:", regex=True)  # 2A / 2B -> 02
    t = pd.to_datetime(s, format="mixed")
    return t.dt.tz_localize(
        "Europe/Berlin",
        ambiguous=~is_second.values,
        nonexistent="shift_forward",
    )

actual_wind = pd.read_csv("actual-wind generation.csv")   # reload fresh
actual_wind = actual_wind.rename(columns={"from": "time_from", "to": "time_to"})

actual_wind["time_from"] = parse_local_time(actual_wind["time_from"]).dt.tz_convert("UTC")
actual_wind["time_to"]   = parse_local_time(actual_wind["time_to"]).dt.tz_convert("UTC")

actual_wind = actual_wind.set_index("time_from").sort_index()

actual_wind.index = actual_wind.index.tz_convert("Europe/Berlin")

print(actual_wind.index.min(), actual_wind.index.max(), len(actual_wind))
print(actual_wind.index.duplicated().sum())   # should be 0


wind_fc = pd.read_csv("wind_forecast.csv")   # reload fresh
wind_fc = wind_fc.rename(columns={"Time from [CET/CEST]": "time_from", "Time to [CET/CEST]": "time_to"})

wind_fc["time_from"] = parse_local_time(wind_fc["time_from"]).dt.tz_convert("UTC")
wind_fc["time_to"]   = parse_local_time(wind_fc["time_to"]).dt.tz_convert("UTC")

wind_fc = wind_fc.set_index("time_from").sort_index()

wind_fc.index = wind_fc.index.tz_convert("Europe/Berlin")

print(wind_fc.index.min(), wind_fc.index.max(), len(wind_fc))
print(wind_fc.index.duplicated().sum())   # should be 0

def parse_to_utc(s):
    s = s.astype(str).str.strip()
    is_epoch = s.str.fullmatch(r"\d{9,}")

    parts = []

    if is_epoch.any():
        # 10 digits = seconds (use unit="ms" if yours have 13 digits)
        parts.append(pd.to_datetime(s[is_epoch].astype("int64"), unit="s", utc=True))

    if (~is_epoch).any():
        txt = s[~is_epoch]
        is_second = txt.str.contains(r"\s2B:")
        t = pd.to_datetime(txt.str.replace(r"2[AB]:", "02:", regex=True), format="mixed")
        t = t.dt.tz_localize("Europe/Berlin",
                             ambiguous=~is_second.values,
                             nonexistent="shift_forward").dt.tz_convert("UTC")
        parts.append(t)

    return pd.concat(parts).reindex(s.index)


day_ahead = pd.read_csv("day-ahead price.csv")
day_ahead = day_ahead.rename(columns={"from": "time_from", "to": "time_to"})

print(day_ahead["time_from"].astype(str).str.fullmatch(r"\d{9,}").sum(), "epoch rows")

day_ahead["time_from"] = parse_to_utc(day_ahead["time_from"])
day_ahead["time_to"]   = parse_to_utc(day_ahead["time_to"])

day_ahead = day_ahead.set_index("time_from").sort_index()

print(day_ahead.index.min(), day_ahead.index.max(), len(day_ahead))
print(day_ahead.index.duplicated().sum())            # should be 0
print(day_ahead.index.to_series().diff().value_counts().head())


2024-06-14 00:00:00+02:00 2026-06-13 23:45:00+02:00 70080
0
2024-06-14 00:00:00+02:00 2026-06-03 23:45:00+02:00 69120
0
34933 epoch rows


DateParseError: Unknown datetime string format, unable to parse: 1.755e+09